In [2]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import io
import base64
import os
import pickle
import traceback

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# ============================================================
# CONFIGURATION
# ============================================================
CONFIDENCE_THRESHOLD = 0.55
LOW_CONFIDENCE_MESSAGE_EN = "⚠️ Inconclusive - Please consult a doctor"

# ============================================================
# IMAGE ENHANCEMENT PIPELINE (NEW)
# ============================================================
def assess_and_enhance_image(image_bgr):
    """
    Detects blur/noise and applies targeted enhancement.
    Args:
        image_bgr (numpy array): OpenCV BGR image.
    Returns:
        enhanced_bgr (numpy array): Enhanced image (BGR).
        action (str): 'none', 'clahe_sharpen', 'denoised'
        quality_score (int): 0-100 overall quality score.
        description (str): Human-readable explanation.
    """
    # Convert to grayscale for analysis
    if len(image_bgr.shape) == 3:
        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    else:
        gray = image_bgr

    # 1. Blur score (Variance of Laplacian)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    # 2. Noise score (standard deviation of difference between image and blurred)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    diff = cv2.absdiff(gray, blurred)
    noise_std = np.std(diff)

    # 3. Normalize scores to 0-100
    blur_score = min(100, max(0, (laplacian_var / 500) * 100))
    noise_score = min(100, max(0, (1 - (noise_std / 50)) * 100))
    overall_quality = int((blur_score * 0.6) + (noise_score * 0.4))

    # 4. Determine action
    enhanced = image_bgr.copy()
    action = "none"
    description = "Image quality is excellent."

    if laplacian_var < 120:  # Blurry / Low contrast
        action = "clahe_sharpen"
        description = "Applied contrast enhancement (CLAHE) and sharpening."

        # Apply CLAHE on L channel for color, else directly on grayscale
        if len(image_bgr.shape) == 3:
            lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            l_enh = clahe.apply(l)
            lab_enh = cv2.merge((l_enh, a, b))
            enhanced = cv2.cvtColor(lab_enh, cv2.COLOR_LAB2BGR)
        else:
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            enhanced = clahe.apply(image_bgr)

        # Unsharp Masking (sharpening)
        kernel = np.array([[-1, -1, -1],
                           [-1,  9, -1],
                           [-1, -1, -1]])
        enhanced = cv2.filter2D(enhanced, -1, kernel)

    elif noise_std > 15:  # High noise (Salt & Pepper / Gaussian)
        action = "denoised"
        description = "Applied edge‑preserving noise reduction (Non‑local Means)."
        if len(image_bgr.shape) == 3:
            enhanced = cv2.fastNlMeansDenoisingColored(image_bgr, None, 10, 10, 7, 21)
        else:
            enhanced = cv2.fastNlMeansDenoising(image_bgr, None, 10, 7, 21)

    return enhanced, action, overall_quality, description

# ============================================================
# LOAD CLASS NAMES
# ============================================================
def load_class_names(pickle_path):
    if os.path.exists(pickle_path):
        with open(pickle_path, "rb") as f:
            names = pickle.load(f)
        print(f"✅ Class names loaded from {pickle_path}: {names}")
        return names
    else:
        default = ['Covid-19', 'Emphysema', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'Tuberculosis']
        print(f"⚠️ {pickle_path} not found. Using default order.")
        return default

CLASS_NAMES_DENSENET = load_class_names("E:\\AI Projects\\code\\class_names.pkl")
CLASS_NAMES_EFFICIENTNET = load_class_names("E:\\AI Projects\\code\\class_names_2.pkl")
CLASS_NAMES_CONVNEXT = load_class_names("E:\\AI Projects\\code\\class_names_3.pkl")

GLOBAL_CLASS_NAMES = CLASS_NAMES_DENSENET
NUM_CLASSES = len(GLOBAL_CLASS_NAMES)
print("✅ Global reference class order:", GLOBAL_CLASS_NAMES)

# ============================================================
# MODEL DEFINITIONS
# ============================================================
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )
    def forward(self, x):
        ca = self.channel_attention(x) * x
        avg_out = torch.mean(ca, dim=1, keepdim=True)
        max_out, _ = torch.max(ca, dim=1, keepdim=True)
        spatial = torch.cat([avg_out, max_out], dim=1)
        sa = self.spatial_attention(spatial) * ca
        return sa

class DenseNetWithCBAM(nn.Module):
    def __init__(self, num_classes=6, dropout_rate=0.5):
        super().__init__()
        base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.cbam = CBAM(channels=1024, reduction=16)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def create_efficientnet(num_classes):
    model = models.efficientnet_b0(weights=None)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.2, inplace=True),
        nn.Linear(num_features, num_classes)
    )
    return model

def create_convnext(num_classes):
    model = models.convnext_tiny(weights=None)
    num_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(num_features, num_classes)
    return model

def load_model(model_class, model_path, num_classes):
    model = model_class(num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✅ Model loaded: {model_path}")
    return model

densenet = load_model(DenseNetWithCBAM, "code\\best_model.pth", len(CLASS_NAMES_DENSENET))
efficientnet = load_model(create_efficientnet, "code\\best_efficientnet.pth", len(CLASS_NAMES_EFFICIENTNET))
convnext = load_model(create_convnext, "code\\best_convnext.pth", len(CLASS_NAMES_CONVNEXT))

# ============================================================
# ENSEMBLE WEIGHTS
# ============================================================
CLASS_WEIGHTS = {
    "densenet": {
        "COVID-19": 0.92, "Emphysema": 1.00, "Normal": 0.99,
        "Pneumonia-Bacterial": 0.99, "Pneumonia-Viral": 0.98, "Tuberculosis": 0.91
    },
    "efficientnet": {
        "COVID-19": 0.9833, "Emphysema": 0.9801, "Normal": 0.9751,
        "Pneumonia-Bacterial": 0.8640, "Pneumonia-Viral": 0.8604, "Tuberculosis": 1.0000
    },
    "convnext": {
        "COVID-19": 0.98, "Emphysema": 0.98, "Normal": 0.98,
        "Pneumonia-Bacterial": 0.92, "Pneumonia-Viral": 0.90, "Tuberculosis": 1.00
    }
}

# ============================================================
# PREPROCESSING
# ============================================================
transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

def preprocess_image(image):
    """
    Expects image as PIL Image or numpy RGB array.
    Returns tensor ready for model input.
    """
    if isinstance(image, Image.Image):
        img = np.array(image.convert('RGB'))
    else:
        # assume numpy array (RGB)
        img = image
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    augmented = transform(image=img)
    return augmented['image'].unsqueeze(0).to(device)

# ============================================================
# PREDICTION FUNCTIONS
# ============================================================
def predict_densenet(tensor):
    with torch.no_grad():
        outputs = densenet(tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    return probs.cpu().numpy().flatten(), CLASS_NAMES_DENSENET

def predict_efficientnet(tensor):
    with torch.no_grad():
        outputs = efficientnet(tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    return probs.cpu().numpy().flatten(), CLASS_NAMES_EFFICIENTNET

def predict_convnext(tensor):
    with torch.no_grad():
        outputs = convnext(tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    return probs.cpu().numpy().flatten(), CLASS_NAMES_CONVNEXT

# ============================================================
# ENSEMBLE
# ============================================================
def ensemble_predict(tensor):
    probs1, names1 = predict_densenet(tensor)
    probs2, names2 = predict_efficientnet(tensor)
    probs3, names3 = predict_convnext(tensor)

    weighted_sum = np.zeros(len(GLOBAL_CLASS_NAMES))
    total_weights = np.zeros(len(GLOBAL_CLASS_NAMES))

    for probs, names, model_key in [
        (probs1, names1, "densenet"),
        (probs2, names2, "efficientnet"),
        (probs3, names3, "convnext")
    ]:
        prob_dict = {name: prob for name, prob in zip(names, probs)}
        for i, global_name in enumerate(GLOBAL_CLASS_NAMES):
            prob = prob_dict.get(global_name, 0.0)
            weight = CLASS_WEIGHTS[model_key].get(global_name, 0.5)
            weighted_sum[i] += prob * weight
            total_weights[i] += weight

    avg_probs = np.divide(weighted_sum, total_weights, out=np.zeros_like(weighted_sum), where=total_weights!=0)

    best_idx = int(np.argmax(avg_probs))
    best_confidence = float(avg_probs[best_idx])
    best_disease = GLOBAL_CLASS_NAMES[best_idx]

    if best_confidence < CONFIDENCE_THRESHOLD:
        return LOW_CONFIDENCE_MESSAGE_EN, best_confidence, "low_confidence_rejection", True

    return best_disease, best_confidence, "class_weighted_soft_voting", False

# ============================================================
# GRAD-CAM
# ============================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_full_backward_hook(backward_hook)

    def generate_heatmap(self, tensor, class_idx=None):
        self.model.eval()
        tensor.requires_grad_(True)
        output = self.model(tensor)

        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()

        self.model.zero_grad()

        one_hot = torch.zeros_like(output)
        one_hot[0][class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        activations = self.activations.detach().cpu().numpy().squeeze()
        gradients = self.gradients.detach().cpu().numpy().squeeze()

        weights = np.mean(gradients, axis=(1, 2))

        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i, :, :]

        cam = np.maximum(cam, 0)
        if np.max(cam) != 0:
            cam = cam / np.max(cam)

        return cam, class_idx

target_densenet = densenet.cbam
target_efficientnet = efficientnet.features[-1]
target_convnext = convnext.features[-1]

gradcam_densenet = GradCAM(densenet, target_densenet)
gradcam_efficientnet = GradCAM(efficientnet, target_efficientnet)
gradcam_convnext = GradCAM(convnext, target_convnext)

print("✅ Grad-CAM models initialized successfully.")

# ============================================================
# FLASK APP
# ============================================================
app = Flask(__name__)
CORS(app)

@app.route('/predict', methods=['POST'])
def predict_endpoint():
    try:
        data = request.get_json()
        if not data or 'image' not in data:
            return jsonify({'error': 'No image provided'}), 400

        # Decode image
        image_data = base64.b64decode(data['image'])
        img = Image.open(io.BytesIO(image_data)).convert('RGB')
        img_np = np.array(img)  # RGB

        # ----- IMAGE ENHANCEMENT (NEW) -----
        # Convert RGB → BGR for OpenCV
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        enhanced_bgr, action, quality_score, description = assess_and_enhance_image(img_bgr)

        # Convert enhanced back to RGB for preprocessing and output
        enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)

        # Encode enhanced image to Base64 (for frontend display)
        enhanced_pil = Image.fromarray(enhanced_rgb)
        buffered = io.BytesIO()
        enhanced_pil.save(buffered, format="PNG")
        enhanced_b64 = base64.b64encode(buffered.getvalue()).decode('utf-8')

        # Preprocess the enhanced image (not the original)
        tensor = preprocess_image(enhanced_rgb)

        # Ensemble prediction on enhanced image
        disease, confidence, decision_source, is_warning = ensemble_predict(tensor)

        # Individual model confidences (for display)
        probs1, _ = predict_densenet(tensor)
        probs2, _ = predict_efficientnet(tensor)
        probs3, _ = predict_convnext(tensor)

        max_conf1 = float(np.max(probs1))
        max_conf2 = float(np.max(probs2))
        max_conf3 = float(np.max(probs3))

        # Build response
        response = {
            'disease': disease,
            'confidence': confidence,
            'decision_source': decision_source,
            'models': {
                'DenseNet121': max_conf1,
                'EfficientNetB0': max_conf2,
                'ConvNeXt': max_conf3
            },
            'warning': is_warning,
            # Enhancement fields
            'enhanced_image': enhanced_b64,
            'quality_score': quality_score,
            'enhancement_action': action,
            'enhancement_description': description
        }

        if is_warning:
            response['explanation'] = f"⚠️ Low confidence ({confidence:.2f}) below threshold ({CONFIDENCE_THRESHOLD}). Please consult a doctor."
        else:
            response['explanation'] = "✅ Result obtained using Class-Weighted Soft Voting on enhanced image."

        return jsonify(response)

    except Exception as e:
        print(f"❌ Error in /predict: {e}")
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/explain', methods=['POST'])
def explain_endpoint():
    try:
        data = request.get_json()
        if not data or 'image' not in data:
            return jsonify({'error': 'No image provided'}), 400

        image_data = base64.b64decode(data['image'])
        img = Image.open(io.BytesIO(image_data)).convert('RGB')
        img_np = np.array(img)

        # Also enhance for Grad-CAM (optional but good for consistency)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        enhanced_bgr, _, _, _ = assess_and_enhance_image(img_bgr)  # we don't need metadata here
        enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)

        tensor = preprocess_image(enhanced_rgb)

        disease, confidence, decision_source, is_warning = ensemble_predict(tensor)

        probs1, _ = predict_densenet(tensor)
        probs2, _ = predict_efficientnet(tensor)
        probs3, _ = predict_convnext(tensor)

        max_conf1 = np.max(probs1)
        max_conf2 = np.max(probs2)
        max_conf3 = np.max(probs3)

        if max_conf2 >= max_conf1 and max_conf2 >= max_conf3:
            best_model_name = "EfficientNet-B0"
            best_gradcam = gradcam_efficientnet
            best_probs = probs2
        elif max_conf3 >= max_conf1 and max_conf3 >= max_conf2:
            best_model_name = "ConvNeXt-Tiny"
            best_gradcam = gradcam_convnext
            best_probs = probs3
        else:
            best_model_name = "DenseNet121+CBAM"
            best_gradcam = gradcam_densenet
            best_probs = probs1

        class_idx = int(np.argmax(best_probs))
        heatmap, _ = best_gradcam.generate_heatmap(tensor, class_idx)

        # Use enhanced image for overlay
        img_for_overlay = cv2.resize(enhanced_rgb, (224, 224))

        heatmap_resized = cv2.resize(heatmap, (224, 224))
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        overlay = cv2.addWeighted(img_for_overlay.astype(np.uint8), 0.6, heatmap_colored.astype(np.uint8), 0.4, 0)

        _, buffer_overlay = cv2.imencode('.png', overlay)
        overlay_base64 = base64.b64encode(buffer_overlay).decode('utf-8')

        _, buffer_heat = cv2.imencode('.png', heatmap_colored)
        heatmap_base64 = base64.b64encode(buffer_heat).decode('utf-8')

        return jsonify({
            'disease': disease,
            'confidence': confidence,
            'overlay_image': overlay_base64,
            'heatmap_only': heatmap_base64,
            'model_used': best_model_name,
            'warning': is_warning,
            'explanation': f"Grad-CAM overlay generated using {best_model_name} (Confidence: {max(max_conf1, max_conf2, max_conf3):.2f})"
        })

    except Exception as e:
        print(f"❌ Error in /explain: {e}")
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'models_loaded': len(CLASS_NAMES_DENSENET),
        'gradcam': 'enabled',
        'encryption': 'disabled',
        'database': 'disabled',
        'image_enhancement': 'enabled'
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

✅ Using device: cpu
✅ Class names loaded from E:\AI Projects\code\class_names.pkl: ['Covid-19', 'Emphysema', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'Tuberculosis']
✅ Class names loaded from E:\AI Projects\code\class_names_2.pkl: ['Covid-19', 'Emphysema', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'Tuberculosis']
✅ Class names loaded from E:\AI Projects\code\class_names_3.pkl: ['Covid-19', 'Emphysema', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'Tuberculosis']
✅ Global reference class order: ['Covid-19', 'Emphysema', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'Tuberculosis']
✅ Model loaded: code\best_model.pth
✅ Model loaded: code\best_efficientnet.pth
✅ Model loaded: code\best_convnext.pth
✅ Grad-CAM models initialized successfully.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.2:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Aug/2026 15:35:53] "OPTIONS /predict HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2026 15:35:53] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2026 15:35:54] "OPTIONS /explain HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2026 15:35:55] "POST /explain HTTP/1.1" 200 -
